# Session 7 — Developing and Deploying APIs for ML Models

**Goal:** go beyond a bare `/predict` route (Session 6's Flask example) and build a
properly designed model API with **FastAPI**: typed request/response schemas,
automatic validation, error handling, versioning, and interactive docs — all fully
runnable in this notebook using FastAPI's `TestClient` (no server process needed).

## Why FastAPI over plain Flask for this

FastAPI uses Python type hints (via Pydantic) to validate incoming requests
automatically and generate interactive API docs for free. A malformed request gets a
clear `422` error with a field-level explanation instead of your model code crashing
on bad input three lines in.

## Prerequisites

```bash
pip install fastapi uvicorn
```
Everything in this notebook runs locally via `TestClient` — no separate server
process or account needed.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import List
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

X, y = load_iris(return_X_y=True)
TARGET_NAMES = load_iris().target_names.tolist()
model = RandomForestClassifier(n_estimators=100, random_state=0).fit(X, y)
print("Model trained. Classes:", TARGET_NAMES)

## Step 1 — Define request/response schemas

Pydantic models are both **documentation** (they show up in the auto-generated docs)
and **validation** (FastAPI rejects anything that doesn't match before your endpoint
code even runs).

In [ ]:
class PredictRequest(BaseModel):
    sepal_length: float = Field(..., gt=0, description="Sepal length in cm")
    sepal_width: float = Field(..., gt=0)
    petal_length: float = Field(..., gt=0)
    petal_width: float = Field(..., gt=0)

class PredictResponse(BaseModel):
    predicted_class: str
    class_probabilities: dict

class BatchPredictRequest(BaseModel):
    instances: List[PredictRequest]

print("Schemas defined.")

## Step 2 — Build the app with versioned, validated endpoints

In [ ]:
app = FastAPI(title="Iris Classifier API", version="1.0.0")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/v1/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    features = np.array([[request.sepal_length, request.sepal_width,
                           request.petal_length, request.petal_width]])
    pred_idx = model.predict(features)[0]
    proba = model.predict_proba(features)[0]

    return PredictResponse(
        predicted_class=TARGET_NAMES[pred_idx],
        class_probabilities={name: round(float(p), 4) for name, p in zip(TARGET_NAMES, proba)},
    )

@app.post("/v1/predict/batch")
def predict_batch(request: BatchPredictRequest):
    if len(request.instances) == 0:
        raise HTTPException(status_code=400, detail="instances list cannot be empty")
    if len(request.instances) > 100:
        raise HTTPException(status_code=400, detail="batch size limited to 100 instances")

    features = np.array([[i.sepal_length, i.sepal_width, i.petal_length, i.petal_width]
                          for i in request.instances])
    preds = model.predict(features)
    return {"predictions": [TARGET_NAMES[p] for p in preds]}

print("App defined with /health, /v1/predict, /v1/predict/batch")

## Step 3 — Test it with `TestClient` (no running server needed)

`TestClient` drives the app in-process — exactly the same routing/validation code
path a real deployed server uses, which makes this the standard way to unit-test a
FastAPI app in CI (Session 10).

In [ ]:
client = TestClient(app)

response = client.get("/health")
print(response.status_code, response.json())

In [ ]:
response = client.post("/v1/predict", json={
    "sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2
})
print(response.status_code)
print(response.json())

## Step 4 — See validation working: send bad input on purpose

In [ ]:
bad_response = client.post("/v1/predict", json={
    "sepal_length": -1.0, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2
})
print("Negative sepal_length ->", bad_response.status_code)
print(bad_response.json())

In [ ]:
missing_field_response = client.post("/v1/predict", json={"sepal_length": 5.1})
print("Missing fields ->", missing_field_response.status_code)
print(missing_field_response.json())

## Step 5 — Batch predictions and the empty-batch error path

In [ ]:
batch_response = client.post("/v1/predict/batch", json={
    "instances": [
        {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2},
        {"sepal_length": 6.7, "sepal_width": 3.1, "petal_length": 4.7, "petal_width": 1.5},
        {"sepal_length": 7.7, "sepal_width": 3.8, "petal_length": 6.7, "petal_width": 2.2},
    ]
})
print(batch_response.status_code, batch_response.json())

empty_batch_response = client.post("/v1/predict/batch", json={"instances": []})
print(empty_batch_response.status_code, empty_batch_response.json())

## Step 6 — Running it for real

`TestClient` is for testing; to actually serve traffic, run it with an ASGI server
(`uvicorn`), which is also exactly what the `CMD` in a Dockerfile (Session 6) would
run inside the container:

In [ ]:
run_command = "uvicorn session7_app:app --host 0.0.0.0 --port 8000"
print(run_command)
print("Once running, visit http://localhost:8000/docs for the auto-generated,")
print("interactive Swagger UI -- built entirely from the Pydantic schemas above.")

## What to try next

* Add an API key check (a FastAPI `Depends()` on a header) so `/v1/predict` requires
  authentication — a minimal step toward production-grade access control.
* Add a `/v2/predict` route with a changed response shape, and see how versioning
  lets you evolve the API without breaking existing clients on `/v1`.
* Wrap this app in the Dockerfile pattern from Session 6 to deploy it as a container.